# 01 - Introducao ao ETS (Error, Trend, Seasonal)

Este notebook apresenta os fundamentos dos modelos de suavizacao exponencial usando a biblioteca **chronobox**.

**Conteudo:**
1. Taxonomia ETS: os 30 modelos possiveis
2. ETS(A,N,N) - Suavizacao Exponencial Simples
3. ETS(A,A,N) - Metodo de Holt (tendencia linear)
4. ETS(A,A,A) - ETS completo com sazonalidade
5. Decomposicao em componentes (nivel, tendencia, sazonalidade)
6. Interpretacao dos parametros de suavizacao
7. Exercicios

## 1. Taxonomia ETS: 30 Modelos

O framework **ETS** (Error, Trend, Seasonal) classifica modelos de suavizacao exponencial por tres componentes:

- **Error (E):** Aditivo (A) ou Multiplicativo (M)
- **Trend (T):** Nenhum (N), Aditivo (A), Aditivo Amortecido (Ad), Multiplicativo (M), Multiplicativo Amortecido (Md)
- **Seasonal (S):** Nenhum (N), Aditivo (A), Multiplicativo (M)

Isso resulta em $2 \times 5 \times 3 = 30$ combinacoes possiveis.

A equacao geral do ETS no espaco de estados:

$$y_t = h(x_{t-1}) + k(x_{t-1}) \varepsilon_t$$

onde $x_t = [l_t, b_t, s_t, \ldots, s_{t-m+1}]'$ e o vetor de estados, $h(\cdot)$ e a funcao de observacao e $k(\cdot)$ controla como o erro entra no sistema.

### Casos Especiais Conhecidos

| Modelo ETS | Nome Classico |
|------------|---------------|
| ETS(A,N,N) | Suavizacao Exponencial Simples (SES) |
| ETS(A,A,N) | Metodo de Holt (tendencia linear) |
| ETS(A,Ad,N) | Metodo de Holt com tendencia amortecida |
| ETS(A,A,A) | Holt-Winters aditivo |
| ETS(M,A,M) | Holt-Winters multiplicativo |
| ETS(A,N,A) | SES com sazonalidade aditiva |

### Tabela Completa dos 30 Modelos ETS

| Error \\ Trend+Season | N,N | N,A | N,M | A,N | A,A | A,M | Ad,N | Ad,A | Ad,M | M,N | M,A | M,M | Md,N | Md,A | Md,M |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| **A** | ANN | ANA | ANM | AAN | AAA | AAM | AdN | AdA | AdM | AMN | AMA | AMM | AMdN | AMdA | AMdM |
| **M** | MNN | MNA | MNM | MAN | MAA | MAM | MdN | MdA | MdM | MMN | MMA | MMM | MMdN | MMdA | MMdM |

**Notacao:** O modelo e identificado como ETS(E, T, S) onde cada letra indica o tipo do componente.

> **Nota:** Nem todas as 30 combinacoes sao estaveis. Modelos com erro multiplicativo e componentes aditivos podem ser problematicos quando a serie tem valores proximos de zero.

## 2. Setup e Carregamento dos Dados

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import sys

from chronobox import ETS
from chronobox.models.ets import get_all_ets_models

np.random.seed(42)

# Adicionar path dos utilitarios
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd())))

print('chronobox importado com sucesso!')
print(f'Total de modelos ETS possiveis: {len(get_all_ets_models())}')

In [ ]:
# Carregar dataset airline (Box-Jenkins, passageiros mensais 1949-1960)
DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), 'data')

airline = pd.read_csv(os.path.join(DATA_DIR, 'airline.csv'), parse_dates=['date'])
airline.set_index('date', inplace=True)
y_airline = airline['passengers'].values

# Carregar dataset sintetico ETS
ets_synth = pd.read_csv(os.path.join(DATA_DIR, 'ets_synthetic.csv'), parse_dates=['date'])
ets_synth.set_index('date', inplace=True)
y_synth = ets_synth['value'].values

print(f'Airline: {len(y_airline)} obs, periodo {airline.index[0].strftime("%Y-%m")} a {airline.index[-1].strftime("%Y-%m")}')
print(f'Sintetico: {len(y_synth)} obs, periodo {ets_synth.index[0].strftime("%Y-%m")} a {ets_synth.index[-1].strftime("%Y-%m")}')

# Visualizacao inicial
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(y_airline, color='steelblue')
axes[0].set_title('Airline Passengers (1949-1960)')
axes[0].set_ylabel('Passageiros (milhares)')
axes[1].plot(y_synth, color='darkorange')
axes[1].set_title('Serie Sintetica ETS(M,A,M)')
axes[1].set_ylabel('Valor')
plt.tight_layout()
plt.show()

## 3. ETS(A,N,N) - Suavizacao Exponencial Simples

O modelo mais simples: **sem tendencia, sem sazonalidade**, apenas erro aditivo.

Equacoes de estado:
$$y_t = l_{t-1} + \varepsilon_t$$
$$l_t = l_{t-1} + \alpha \varepsilon_t$$

O parametro $\alpha \in (0,1)$ controla a velocidade de adaptacao:
- $\alpha \to 0$: previsao quase constante (media historica)
- $\alpha \to 1$: previsao = ultima observacao (random walk)

In [ ]:
# Ajustar ETS(A,N,N) - Suavizacao Exponencial Simples
model_ann = ETS(error='A', trend='N', seasonal='N')
res_ann = model_ann.fit(y_airline)

print(res_ann.summary())
print(f'\nParametro alpha: {res_ann.alpha:.4f}')
print(f'Nivel inicial (l0): {res_ann.l0:.2f}')
print(f'AIC: {res_ann.aic:.2f}')

In [ ]:
# Grafico 1: Ajuste ETS(A,N,N)
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(y_airline, label='Observado', color='steelblue', linewidth=1.5)
ax.plot(res_ann.fittedvalues, label='ETS(A,N,N) ajustado', color='darkorange',
        linewidth=1.2, linestyle='--')
ax.set_title('ETS(A,N,N) - Suavizacao Exponencial Simples')
ax.set_xlabel('Observacao')
ax.set_ylabel('Passageiros')
ax.legend()
plt.tight_layout()
plt.show()

print('Nota: O ETS(A,N,N) nao captura tendencia nem sazonalidade.')
print('A previsao sera uma linha constante (nivel atual).')

## 4. ETS(A,A,N) - Metodo de Holt (Tendencia Linear)

Adiciona um componente de **tendencia aditiva**:

$$y_t = l_{t-1} + b_{t-1} + \varepsilon_t$$
$$l_t = l_{t-1} + b_{t-1} + \alpha \varepsilon_t$$
$$b_t = b_{t-1} + \beta \varepsilon_t$$

Dois parametros de suavizacao:
- $\alpha$: controla o nivel
- $\beta$: controla a tendencia

In [ ]:
# Ajustar ETS(A,A,N) - Holt com tendencia linear
model_aan = ETS(error='A', trend='A', seasonal='N')
res_aan = model_aan.fit(y_airline)

print(res_aan.summary())
print(f'\nalpha (nivel): {res_aan.alpha:.4f}')
print(f'beta (tendencia): {res_aan.beta:.4f}')
print(f'Nivel inicial (l0): {res_aan.l0:.2f}')
print(f'Tendencia inicial (b0): {res_aan.b0:.2f}')
print(f'AIC: {res_aan.aic:.2f}')

In [ ]:
# Grafico 2: Comparacao ETS(A,N,N) vs ETS(A,A,N)
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(y_airline, label='Observado', color='steelblue', linewidth=1.5)
ax.plot(res_ann.fittedvalues, label=f'ETS(A,N,N) AIC={res_ann.aic:.1f}',
        color='darkorange', linewidth=1.2, linestyle='--')
ax.plot(res_aan.fittedvalues, label=f'ETS(A,A,N) AIC={res_aan.aic:.1f}',
        color='green', linewidth=1.2, linestyle='--')
ax.set_title('Comparacao: sem tendencia vs com tendencia')
ax.set_xlabel('Observacao')
ax.set_ylabel('Passageiros')
ax.legend()
plt.tight_layout()
plt.show()

print('O ETS(A,A,N) captura a tendencia crescente, mas ainda ignora a sazonalidade.')

## 5. ETS(A,A,A) - Modelo Completo com Sazonalidade

Adiciona **sazonalidade aditiva** ao modelo de Holt:

$$y_t = l_{t-1} + b_{t-1} + s_{t-m} + \varepsilon_t$$
$$l_t = l_{t-1} + b_{t-1} + \alpha \varepsilon_t$$
$$b_t = b_{t-1} + \beta \varepsilon_t$$
$$s_t = s_{t-m} + \gamma \varepsilon_t$$

Tres parametros de suavizacao:
- $\alpha$: nivel
- $\beta$: tendencia
- $\gamma$: sazonalidade
- $m$: periodo sazonal (12 para dados mensais)

In [ ]:
# Ajustar ETS(A,A,A) - Modelo completo com sazonalidade aditiva
model_aaa = ETS(error='A', trend='A', seasonal='A', seasonal_period=12)
res_aaa = model_aaa.fit(y_airline)

print(res_aaa.summary())
print(f'\nalpha (nivel): {res_aaa.alpha:.4f}')
print(f'beta (tendencia): {res_aaa.beta:.4f}')
print(f'gamma (sazonalidade): {res_aaa.gamma:.4f}')
print(f'AIC: {res_aaa.aic:.2f}')

In [ ]:
# Grafico 3: Ajuste ETS(A,A,A) com componentes
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

# Serie observada e ajustada
axes[0].plot(y_airline, label='Observado', color='steelblue')
axes[0].plot(res_aaa.fittedvalues, label='Ajustado', color='darkorange', linestyle='--')
axes[0].set_title('ETS(A,A,A) - Airline Passengers')
axes[0].legend()
axes[0].set_ylabel('Passageiros')

# Componente de nivel
states = res_aaa.states
n = len(y_airline)
axes[1].plot(states[:n, 0], color='green')
axes[1].set_title('Componente: Nivel ($l_t$)')
axes[1].set_ylabel('Nivel')

# Componente de tendencia
axes[2].plot(states[:n, 1], color='purple')
axes[2].set_title('Componente: Tendencia ($b_t$)')
axes[2].set_ylabel('Tendencia')

# Componente sazonal
axes[3].plot(states[:n, 2], color='crimson')
axes[3].set_title('Componente: Sazonalidade ($s_t$)')
axes[3].set_ylabel('Sazonal')
axes[3].set_xlabel('Observacao')

plt.tight_layout()
plt.show()

print('A decomposicao mostra:')
print('- Nivel crescente (tendencia de longo prazo)')
print('- Tendencia positiva (crescimento)')
print('- Sazonalidade com picos nos meses de verao')

## 6. Interpretacao dos Parametros de Suavizacao

Os parametros $\alpha$, $\beta$ e $\gamma$ controlam a **velocidade de adaptacao** de cada componente:

| Parametro | Componente | Proximo de 0 | Proximo de 1 |
|-----------|-----------|--------------|--------------|
| $\alpha$ | Nivel | Nivel muda lentamente | Nivel reage rapido |
| $\beta$ | Tendencia | Tendencia estavel | Tendencia volatil |
| $\gamma$ | Sazonalidade | Padrao sazonal fixo | Sazonalidade muda rapido |

> **Regra pratica:** Valores muito altos de $\beta$ ou $\gamma$ podem indicar overfitting. Em geral, $\beta < \alpha$ e $\gamma < \alpha$.

In [ ]:
# Comparacao de todos os modelos ajustados
print(f'{"Modelo":<15} {"alpha":>8} {"beta":>8} {"gamma":>8} {"AIC":>10} {"BIC":>10}')
print('-' * 55)

modelos = {
    'ETS(A,N,N)': res_ann,
    'ETS(A,A,N)': res_aan,
    'ETS(A,A,A)': res_aaa,
}

for nome, res in modelos.items():
    alpha = f'{res.alpha:.4f}'
    beta = f'{res.beta:.4f}' if res.beta is not None else '   -   '
    gamma = f'{res.gamma:.4f}' if res.gamma is not None else '   -   '
    print(f'{nome:<15} {alpha:>8} {beta:>8} {gamma:>8} {res.aic:>10.2f} {res.bic:>10.2f}')

melhor = min(modelos, key=lambda k: modelos[k].aic)
print(f'\nMelhor modelo por AIC: {melhor}')
print('\nInterpretacao dos parametros do melhor modelo:')
best = modelos[melhor]
print(f'  alpha={best.alpha:.4f}: nivel se adapta {"rapido" if best.alpha > 0.5 else "moderadamente" if best.alpha > 0.2 else "lentamente"}')
if best.beta is not None:
    print(f'  beta={best.beta:.4f}: tendencia {"volatil" if best.beta > 0.3 else "estavel"}')
if best.gamma is not None:
    print(f'  gamma={best.gamma:.4f}: sazonalidade {"mutavel" if best.gamma > 0.3 else "estavel"}')

## 7. Diagnostico de Residuos

In [ ]:
# Grafico 4: Diagnostico de residuos do melhor modelo
resid = res_aaa.resid

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Residuos ao longo do tempo
axes[0, 0].plot(resid, color='steelblue', linewidth=0.8)
axes[0, 0].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[0, 0].set_title('Residuos ao Longo do Tempo')
axes[0, 0].set_ylabel('Residuo')

# Histograma dos residuos
axes[0, 1].hist(resid, bins=20, color='steelblue', edgecolor='white', density=True)
x = np.linspace(resid.min(), resid.max(), 100)
from scipy.stats import norm
axes[0, 1].plot(x, norm.pdf(x, resid.mean(), resid.std()), 'r-', linewidth=2)
axes[0, 1].set_title('Histograma dos Residuos')

# ACF dos residuos
max_lag = 24
n = len(resid)
acf = np.array([np.corrcoef(resid[:n-k], resid[k:])[0, 1] for k in range(max_lag + 1)])
acf[0] = 1.0
ci = 1.96 / np.sqrt(n)
axes[1, 0].bar(range(max_lag + 1), acf, width=0.3, color='steelblue')
axes[1, 0].axhline(ci, color='red', linestyle='--', alpha=0.7)
axes[1, 0].axhline(-ci, color='red', linestyle='--', alpha=0.7)
axes[1, 0].set_title('ACF dos Residuos')

# QQ-plot
from scipy.stats import probplot
probplot(resid, dist='norm', plot=axes[1, 1])
axes[1, 1].set_title('Q-Q Plot')

plt.suptitle('Diagnostico de Residuos - ETS(A,A,A)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 8. Previsao

In [ ]:
# Previsao 24 meses a frente com ETS(A,A,A)
fc = res_aaa.forecast(steps=24)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(range(len(y_airline)), y_airline, label='Observado', color='steelblue')
ax.plot(range(len(y_airline)), res_aaa.fittedvalues, label='Ajustado',
        color='darkorange', linestyle='--', alpha=0.7)
ax.plot(range(len(y_airline), len(y_airline) + 24), fc,
        label='Previsao ETS(A,A,A)', color='crimson', linewidth=2)
ax.axvline(len(y_airline) - 1, color='gray', linestyle=':', alpha=0.5)
ax.set_title('ETS(A,A,A) - Previsao 24 Meses')
ax.set_xlabel('Observacao')
ax.set_ylabel('Passageiros')
ax.legend()
plt.tight_layout()
plt.show()

print('Previsoes:')
for i in range(0, 24, 6):
    print(f'  Passo {i+1:2d}: {fc[i]:.1f}')

## Exercicio

### Exercicio 1: Identificar o modelo ETS adequado

Usando o dataset `airline.csv`:

1. Ajuste os modelos ETS(A,N,N), ETS(A,A,N), ETS(A,A,A), ETS(M,A,M) e ETS(M,Ad,M)
2. Compare AIC e BIC de cada modelo
3. Qual modelo melhor captura a sazonalidade multiplicativa dos dados?
4. Plote os ajustes sobrepostos e justifique sua escolha

**Dica:** A amplitude da sazonalidade do airline cresce com o nivel - isso sugere sazonalidade multiplicativa!

In [ ]:
# Exercicio 1 - Seu codigo aqui
# model_mam = ETS(error='M', trend='A', seasonal='M', seasonal_period=12)
# res_mam = model_mam.fit(y_airline)
# ...

## Exercicio

### Exercicio 2: ETS no dataset sintetico

Usando o dataset `ets_synthetic.csv`:

1. Plote a serie e identifique visualmente tendencia e tipo de sazonalidade
2. Ajuste ETS(A,A,A) e ETS(M,A,M) e compare os resultados
3. Analise os residuos do melhor modelo
4. O dataset foi gerado como ETS(M,A,M) - seu modelo conseguiu identificar isso?

In [ ]:
# Exercicio 2 - Seu codigo aqui
# model = ETS(error='M', trend='A', seasonal='M', seasonal_period=12)
# res = model.fit(y_synth)
# ...